# Finalized Stage 1 Autoencoders: Multi-Source ALE Maps

This notebook runs the fixed multi-source Stage 1 pipeline that defines the four AE branches evaluated downstream in Notebook 4:

1. Train the mixed-source baseline autoencoder with raw MSE.
2. Fine-tune that checkpoint separately on PubMed, Nilearn, and NeuroVault.
3. Evaluate the selected checkpoint for each of the four autoencoders and write one combined metrics table.

Completed runs in the current output directory are reused automatically. There are no architecture, loss, data-source, or checkpoint-selection alternatives in this notebook. Notebook 4 currently resolves locked copies of these four AE branches through `neurovlm.retrieval_resources`; it does not automatically consume the newest registry written here.


In [ ]:
from pathlib import Path
import csv
import json
import os
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

IN_COLAB = "google.colab" in sys.modules
default_repo_dir = Path.cwd() if (Path.cwd() / "experiments/3dcnn/atlas_free_cnn").exists() else Path("/content/neurovlm")
REPO_URL = os.environ.get("NEUROVLM_REPO_URL", "https://github.com/neurovlm/neurovlm.git")
REPO_BRANCH = os.environ.get("NEUROVLM_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("NEUROVLM_REPO_DIR", str(default_repo_dir))).expanduser()
DRIVE_ROOT = Path(os.environ.get("NEUROVLM_DRIVE_ROOT", "/content/drive/MyDrive/neurovlm")).expanduser()
INSTALL_DEPENDENCIES = os.environ.get("NEUROVLM_INSTALL_DEPS", "1" if IN_COLAB else "0") == "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


def run_cmd(cmd, cwd=None, *, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        run_cmd(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    elif not (REPO_DIR / ".git").exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a git checkout")
    else:
        run_cmd(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
        run_cmd(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
        run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])
elif not (REPO_DIR / "experiments/3dcnn/atlas_free_cnn").exists():
    raise RuntimeError("Set NEUROVLM_REPO_DIR to the local neurovlm checkout before running this notebook")

os.chdir(REPO_DIR)
if INSTALL_DEPENDENCIES:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz,notebook,metrics]"])

for path in [REPO_DIR / "experiments/3dcnn", REPO_DIR / "src", REPO_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Working directory:", Path.cwd())
print("Repo branch:", run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, check=False).stdout.strip())
print("Output root:", DRIVE_ROOT)

from atlas_free_cnn.notebook_utils import discover_default_unified_split_dir
from atlas_free_cnn.pipeline_outputs import write_table
from atlas_free_cnn.training.train_autoencoder import (
    evaluate_saved_checkpoints_from_config,
    train_from_config,
    train_stage1b_from_config,
)


In [ ]:
HF_DATASET_REPO = os.environ.get("NEUROVLM_ATLAS_FREE_HF_REPO", "neurovlm/atlas_free_cnn_dataset")
UNIFIED_SPLIT_DIR = discover_default_unified_split_dir(
    repo_dir=REPO_DIR,
    drive_root=DRIVE_ROOT,
    dataset_repo=HF_DATASET_REPO,
)
TRAIN_JSONL = str(UNIFIED_SPLIT_DIR / "train.jsonl")
VAL_JSONL = str(UNIFIED_SPLIT_DIR / "val.jsonl")
TEST_JSONL = str(UNIFIED_SPLIT_DIR / "test.jsonl")
print("Unified split directory:", UNIFIED_SPLIT_DIR)

MIXED_VARIANT = "mixed_baseline_raw_mse"
BASELINE_CHECKPOINT_SELECTION = "best_val_loss"
DOMAIN_CHECKPOINT_SELECTION = "best_top5_dice"
STAGE1B_SPECS = (
    {"mode": "mixed_pretrain_to_pubmed", "domain": "pubmed", "variant": "mixed_baseline_to_pubmed", "registry_key": "mixed_to_pubmed_stage1b"},
    {"mode": "mixed_pretrain_to_nilearn", "domain": "nilearn", "variant": "mixed_baseline_to_nilearn", "registry_key": "mixed_to_nilearn_stage1b"},
    {"mode": "mixed_pretrain_to_neurovault", "domain": "neurovault", "variant": "mixed_baseline_to_neurovault", "registry_key": "mixed_to_neurovault_stage1b"},
)

AE_EPOCHS = 300
STAGE1B_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
AE_BATCH_CANDIDATES = [512, 384, 256, 192, 128, 96, 64, 48, 32, 16]
STAGE1B_BATCH_CANDIDATES = [96, 64, 48, 32, 16]
AE_PREFLIGHT_RESERVE_GB = 12.0
STAGE1B_PREFLIGHT_RESERVE_GB = 28.0
NUM_WORKERS = int(os.environ.get("NEUROVLM_NUM_WORKERS", "4" if IN_COLAB else "0"))
EVAL_NUM_WORKERS = int(os.environ.get("NEUROVLM_EVAL_NUM_WORKERS", str(NUM_WORKERS)))
PREFETCH_FACTOR = int(os.environ.get("NEUROVLM_PREFETCH_FACTOR", "4"))
METRICS_DEVICE = os.environ.get("NEUROVLM_METRICS_DEVICE", "cuda")

OUTPUT_ROOT = DRIVE_ROOT / "runs_atlas_free_cnn_stage1"
RUN_DIR = Path(os.environ.get("NEUROVLM_STAGE1_RUN_DIR", str(OUTPUT_ROOT / "finalized_stage1"))).expanduser()
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Run directory:", RUN_DIR)
print({
    "mixed_epochs": AE_EPOCHS,
    "domain_finetuning_epochs": STAGE1B_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "baseline_checkpoint_selection": BASELINE_CHECKPOINT_SELECTION,
    "domain_checkpoint_selection": DOMAIN_CHECKPOINT_SELECTION,
    "num_workers": NUM_WORKERS,
    "metrics_device": METRICS_DEVICE,
})


In [ ]:
import gc
import torch


def clear_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def is_cuda_oom(exc: BaseException) -> bool:
    text = str(exc).lower()
    return isinstance(exc, torch.cuda.OutOfMemoryError) or "cuda out of memory" in text or "outofmemoryerror" in text


def train_with_batch_fallback(base_cfg, candidates, *, reserve_gb, train_fn):
    candidates = [int(value) for value in candidates]
    last_exc = None
    for max_batch in candidates:
        clear_cuda_memory()
        cfg = dict(base_cfg)
        cfg["batch_candidates"] = [value for value in candidates if value <= max_batch]
        cfg["max_batch_size"] = max_batch
        cfg["batch_size"] = min(int(cfg.get("batch_size", 64)), max_batch)
        cfg["preflight_vram_reserve_gb"] = reserve_gb
        print(f"Trying max AE batch size {max_batch}; candidates={cfg['batch_candidates']}")
        try:
            return train_fn(cfg)
        except Exception as exc:
            if not is_cuda_oom(exc):
                raise
            last_exc = exc
            print(f"OOM at max_batch_size={max_batch}; retrying with a smaller candidate")
    raise RuntimeError(f"All AE batch candidates failed: {candidates}") from last_exc


def ensure_selected_checkpoint_metrics(run_dir: Path) -> Path:
    metrics_path = run_dir / "metrics/reconstruction_summary_by_source.csv"
    checkpoint_metrics_path = run_dir / "metrics/reconstruction_summary_by_checkpoint_source.csv"
    if metrics_path.exists() or checkpoint_metrics_path.exists():
        return metrics_path if metrics_path.exists() else checkpoint_metrics_path
    config_path = run_dir / "autoencoder_config.json"
    if not config_path.exists():
        config_path = run_dir / "config/ae_config.json"
    if not config_path.exists():
        raise FileNotFoundError(f"Cannot evaluate {run_dir}: its training configuration is missing")
    evaluate_saved_checkpoints_from_config(config_path)
    return checkpoint_metrics_path


def completed_checkpoint(run_dir: Path, checkpoint_name: str) -> Path | None:
    checkpoint = run_dir / "checkpoints" / f"{checkpoint_name}.pt"
    if checkpoint.exists() and (run_dir / "training_stop.json").exists():
        ensure_selected_checkpoint_metrics(run_dir)
        return checkpoint
    return None


common_config = {
    "train_jsonl": TRAIN_JSONL,
    "val_jsonl": VAL_JSONL,
    "test_jsonl": TEST_JSONL,
    "target_shape": [36, 45, 38],
    "model": {"latent_dim": 384, "base_channels": 64, "num_blocks": 4, "dropout": 0.1, "norm": "group", "pooling": "max"},
    "amp": True,
    "gradient_clipping": 1.0,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "final_eval": True,
    "eval_all_checkpoints": False,
    "save_plots": True,
    "progress": True,
    "num_workers": NUM_WORKERS,
    "eval_num_workers": EVAL_NUM_WORKERS,
    "prefetch_factor": PREFETCH_FACTOR,
    "pin_memory": True,
    "persistent_workers": NUM_WORKERS > 0,
    "metrics_device": METRICS_DEVICE,
    "compute_epoch_source_metrics": False,
    "compute_train_metrics": False,
    "train_metric_batches": 0,
    "val_metric_batches": 16,
}

stage1a_out_dir = RUN_DIR / "01_stage1_ae_pretraining" / MIXED_VARIANT
stage1a_checkpoint = completed_checkpoint(stage1a_out_dir, BASELINE_CHECKPOINT_SELECTION)
if stage1a_checkpoint is None:
    stage1a_config = {
        **common_config,
        "output_dir": str(stage1a_out_dir),
        "checkpoint_dir": str(stage1a_out_dir / "checkpoints"),
        "data_mode": "mixed",
        "ae_variant": MIXED_VARIANT,
        "ae_training_recipe": "baseline_raw_mse",
        "source_sampling": "natural",
        "loss": {"type": "raw_mse", "lambda_foreground": 0.0, "lambda_topk": 0.0, "prediction_activation": "none"},
        "checkpoint_selection_metric": BASELINE_CHECKPOINT_SELECTION,
        "eval_checkpoint_names": [BASELINE_CHECKPOINT_SELECTION],
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "batch_size": 64,
        "epochs": AE_EPOCHS,
    }
    stage1a_result = train_with_batch_fallback(
        stage1a_config,
        AE_BATCH_CANDIDATES,
        reserve_gb=AE_PREFLIGHT_RESERVE_GB,
        train_fn=train_from_config,
    )
    stage1a_checkpoint = Path(stage1a_result["best_checkpoint"])
    stage1a_status = "trained"
else:
    stage1a_status = "reused_completed_run"
    print("Reusing completed mixed baseline:", stage1a_checkpoint)

results = [{
    "registry_key": "mixed_stage1a",
    "ae_variant": MIXED_VARIANT,
    "training_domain": "mixed",
    "selection": BASELINE_CHECKPOINT_SELECTION,
    "checkpoint_dir": str(stage1a_checkpoint.parent),
    "best_checkpoint": str(stage1a_checkpoint),
    "status": stage1a_status,
}]
clear_cuda_memory()
results


In [ ]:
for spec in STAGE1B_SPECS:
    out_dir = RUN_DIR / "02_stage1b_ae_finetuning" / spec["domain"]
    checkpoint = completed_checkpoint(out_dir, DOMAIN_CHECKPOINT_SELECTION)
    if checkpoint is None:
        config = {
            **common_config,
            "stage1b_mode": spec["mode"],
            "mixed_pretrain_checkpoint": str(stage1a_checkpoint),
            "output_dir": str(out_dir),
            "checkpoint_dir": str(out_dir / "checkpoints"),
            "ae_variant": spec["variant"],
            "stage1b_seed_variant": MIXED_VARIANT,
            "stage1b_seed_checkpoint": str(stage1a_checkpoint),
            "checkpoint_selection_metric": DOMAIN_CHECKPOINT_SELECTION,
            "eval_checkpoint_names": [DOMAIN_CHECKPOINT_SELECTION],
            "lr": 1e-4,
            "weight_decay": 1e-4,
            "freeze_mode": "none",
            "epochs": STAGE1B_EPOCHS,
            "batch_size": 64,
            "loss": {"type": "raw_mse", "lambda_foreground": 0.0, "lambda_topk": 0.0, "prediction_activation": "none"},
        }
        result = train_with_batch_fallback(
            config,
            STAGE1B_BATCH_CANDIDATES,
            reserve_gb=STAGE1B_PREFLIGHT_RESERVE_GB,
            train_fn=train_stage1b_from_config,
        )
        checkpoint = Path(result["best_checkpoint"])
        status = "trained"
    else:
        status = "reused_completed_run"
        print(f"Reusing completed {spec['domain']} fine-tune: {checkpoint}")

    results.append({
        "registry_key": spec["registry_key"],
        "ae_variant": spec["variant"],
        "training_domain": spec["domain"],
        "selection": DOMAIN_CHECKPOINT_SELECTION,
        "checkpoint_dir": str(checkpoint.parent),
        "best_checkpoint": str(checkpoint),
        "seed_checkpoint": str(stage1a_checkpoint),
        "status": status,
    })
    clear_cuda_memory()

assert len(results) == 4, f"Expected four finalized AE checkpoints, found {len(results)}"
results


In [ ]:
metadata_dir = RUN_DIR / "00_run_metadata"
final_dir = RUN_DIR / "07_final_comparison"
metadata_dir.mkdir(parents=True, exist_ok=True)
final_dir.mkdir(parents=True, exist_ok=True)

checkpoint_registry = {
    row["registry_key"]: {
        "path": row["best_checkpoint"],
        "ae_variant": row["ae_variant"],
        "training_domain": row["training_domain"],
        "selection": row["selection"],
    }
    for row in results
}
(final_dir / "selected_ae_checkpoints.json").write_text(json.dumps(checkpoint_registry, indent=2) + "\n")
write_table(metadata_dir / "stage1_status.csv", results)

summary_rows = []
for result in results:
    run_dir = Path(result["checkpoint_dir"]).parent
    candidates = [
        run_dir / "metrics/reconstruction_summary_by_source.csv",
        run_dir / "metrics/reconstruction_summary_by_checkpoint_source.csv",
    ]
    metrics_path = next((path for path in candidates if path.exists()), None)
    if metrics_path is None:
        raise FileNotFoundError(f"No reconstruction metrics found for {result['ae_variant']}")
    with metrics_path.open(newline="") as handle:
        for metric in csv.DictReader(handle):
            if metric.get("split") not in {"val", "test"}:
                continue
            if metric.get("source_detail") != "ALL_DETAILS":
                continue
            if metric.get("source") not in {"pubmed", "nilearn", "neurovault"}:
                continue
            if metric.get("checkpoint") and metric["checkpoint"] != result["selection"]:
                continue
            summary_rows.append({
                "ae_variant": result["ae_variant"],
                "training_domain": result["training_domain"],
                "selected_checkpoint": result["best_checkpoint"],
                "selection_metric": result["selection"],
                "split": metric["split"],
                "evaluation_source": metric["source"],
                "n": metric.get("n", ""),
                "spatial_corr": metric.get("spatial_corr", ""),
                "top1_dice": metric.get("top1_dice", ""),
                "top5_dice": metric.get("top5_dice", ""),
                "top10_dice": metric.get("top10_dice", ""),
                "mse": metric.get("mse", metric.get("reconstruction_mse", "")),
                "foreground_mse": metric.get("foreground_mse", ""),
                "metrics_path": str(metrics_path),
            })

write_table(final_dir / "final_summary_table.csv", summary_rows)
print("Selected checkpoint registry:", final_dir / "selected_ae_checkpoints.json")
print("Combined reconstruction metrics:", final_dir / "final_summary_table.csv")
summary_rows
